In [560]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Import Statsmodels
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.tools.eval_measures import rmse, aic

In [561]:
df_vitals = pd.read_csv('vitals_combined_interpolated.csv')
df_beta = pd.read_csv('beta-blocker-dosages.csv')

In [562]:
df_vitals.columns = ['patient_uuid', 'timestamp', 'batch', 'HR', 'SBP']

In [563]:
df_vitals.head()

,patient_uuid,timestamp,batch,HR,SBP
0,1127912,2019-11-19,0,87.261064,124.85394
1,1127912,2019-11-20,0,92.000000,120.50000
2,1127912,2019-11-21,0,87.000000,131.75000
3,1127912,2019-11-22,0,89.000000,127.00000
4,1127912,2019-11-23,0,93.000000,109.50000


In [711]:
df_vitals['timestamp'] = pd.to_datetime(df_vitals['timestamp'], errors='coerce')

df_vitals = df_vitals[['patient_uuid', 'timestamp', 'batch', 'HR', 'SBP']]

df_vitals = df_vitals[(df_vitals.timestamp >= '2018-10-01') & (df_vitals.timestamp <= '2020-01-14')]
df_vitals = df_vitals[~df_vitals['SBP'].isna()]
df_vitals = df_vitals[~df_vitals['HR'].isna()]

# columns with number of non-empty rows
(~df_vitals.isna()).sum() / len(df_vitals) * 100

patient_uuid    100.0
timestamp       100.0
batch           100.0
HR              100.0
SBP             100.0
dtype: float64

In [565]:
len(df_vitals)

127635

In [568]:
# Drops batches containing less than 30 observations.
df_recordings_per_patient = df_vitals.groupby(['patient_uuid', 'batch']).count().reset_index()
low_batches = df_recordings_per_patient[(df_recordings_per_patient.HR <= 30)][['patient_uuid','batch']]

for i in range(len(low_batches)):
    low_patient = low_batches.patient_uuid.iloc[i]
    low_batch = low_batches.batch.iloc[i]
    df_vitals.drop(df_vitals[(df_vitals.patient_uuid == low_patient) & (df_vitals.batch == low_batch)].index, inplace=True)

In [570]:
# Creates patient list based on the cleaned dataframe.
patients = df_vitals.patient_uuid.unique()

In [486]:
# Creates a temporary time series for a patient for testing functions and debugging.
df_temp1 = df_vitals[df_vitals.patient_uuid == 36279955] #73939445, 36279955, 62596157
df_temp1.index = pd.DatetimeIndex(df_temp1.timestamp).to_period('D')
df_temp1 = df_temp1[['HR','SBP']].copy()
df_temp1.head()

,HR,SBP
timestamp,,
2018-10-01,73.333333,148.666667
2018-10-02,80.000000,168.000000
2018-10-03,80.000000,176.000000
2018-10-04,74.000000,179.000000
2018-10-05,72.000000,135.000000


In [487]:
from statsmodels.tsa.stattools import grangercausalitytests
maxlag=12
test = 'ssr_chi2test'
def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False):    
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data     : pandas dataframe containing the time series variables
    variables: list containing names of the time series variables.
    
    output   : dataframe containing p-values to test causation.
    """
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r, c]], maxlag=maxlag, verbose=False)
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(maxlag)]
            if verbose: print(f'Y = {r}, X = {c}, P Values = {p_values}')
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df



In [488]:
grangers_causation_matrix(df_temp1, variables = df_temp1.columns)

,HR_x,SBP_x
HR_y,1.0000,0.0013
SBP_y,0.0699,1.0000


In [362]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

def cointegration_test(df, alpha=0.05): 
    """
    Perform Johanson's Cointegration Test and Report Summary.
    
    df    : time series dataframe for a particular patient.
    
    output: prints summary of the results of the cointegration test.
    """
    out = coint_johansen(df,-1,5)
    d = {'0.90':0, '0.95':1, '0.99':2}
    traces = out.lr1
    cvts = out.cvt[:, d[str(1-alpha)]]
    def adjust(val, length= 6): return str(val).ljust(length)

    # Summary
    print('Name   ::  Test Stat > C(95%)    =>   Signif  \n', '--'*20)
    for col, trace, cvt in zip(df.columns, traces, cvts):
        print(adjust(col), ':: ', adjust(round(trace,2), 9), ">", adjust(cvt, 8), ' =>  ' , trace > cvt)



In [489]:
cointegration_test(df_temp1)

Name   ::  Test Stat > C(95%)    =>   Signif  
 ----------------------------------------
HR     ::  16.94     > 12.3212   =>   True
SBP    ::  0.08      > 4.1296    =>   False


In [490]:
nobs = 7
df_train, df_test = df_temp1[0:-nobs], df_temp1[-nobs:]

In [491]:
def adfuller_test(series, signif=0.05, name='', verbose=False):
    """
    Performs ADFuller to test for Stationarity of given series and prints report.
    
    series: pandas series of the feature to test stationarity on.
    
    output: True if stationary, False if not stationary.
    
    """
    r = adfuller(series, autolag='AIC')
    p_value = round(r[1], 4)

    if p_value <= signif:
        is_stationary = True
    else:
        is_stationary = False
        
    return is_stationary

In [492]:
# ADF Test on each column
for name, column in df_train.iteritems():
    print(adfuller_test(column, name=column.name))
    

False
True


In [493]:
# Differenced time series since HR is not stationary.
df_differenced = df_train.diff().dropna()
df_differenced.head()

,HR,SBP
timestamp,,
2018-10-02,6.666667,19.333333
2018-10-03,0.000000,8.000000
2018-10-04,-6.000000,3.000000
2018-10-05,-2.000000,-44.000000
2018-10-06,8.000000,25.000000


In [494]:
# ADF Test on each column
for name, column in df_differenced.iteritems():
    print(adfuller_test(column, name=column.name))

True
True


In [495]:
# fits model based on the order that yields the lowest AIC and prints summary.
model = VAR(df_differenced)
x = model.select_order(maxlags=20)
lag_order = x.aic
model_fitted = model.fit(lag_order)
model_fitted.summary()

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 03, Mar, 2020
Time:                     18:12:22
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                    8.86695
Nobs:                     452.000    HQIC:                   8.65739
Log likelihood:          -3170.49    FPE:                    5020.10
AIC:                      8.52111    Det(Omega_mle):         4623.25
--------------------------------------------------------------------
Results for equation HR
            coefficient       std. error           t-stat            prob
-------------------------------------------------------------------------
const         -0.040447         0.258661           -0.156           0.876
L1.HR         -0.737663         0.049018          -15.049           0.000
L1.SBP        -0.011350         0.021167           -0.536           0.592
L2.HR         -0.

Performs the durbin watson statistic to check for serial correlation of residuals to ensure that the model is able to explain the variances and patterns in the time series.
The value of this statistic can vary between 0 and 4. The closer it is to the value 2, then there is no significant serial correlation. The closer to 0, there is a positive serial correlation, and the closer it is to 4 implies negative serial correlation.

In [496]:
from statsmodels.stats.stattools import durbin_watson
out = durbin_watson(model_fitted.resid)

for col, val in zip(df_temp1.columns, out):
    print(col, ':', round(val, 2))

HR : 2.0
SBP : 2.01


In [497]:
# Input data for forecasting
#forecast_input = df_train.values[-lag_order:]
forecast_input = df_differenced.values[-lag_order:]
forecast_input

array([[  4.91666667, -15.41666667],
       [-12.16666667,   4.16666667],
       [ 10.5       ,   7.5       ],
       [  0.        ,   3.5       ],
       [ -1.80337549,  -9.56717035],
       [ -5.86329118,   3.06717035],
       [  9.66666667,   9.5       ],
       [-10.66666667,  -8.83333333],
       [ 10.16666667,   0.33333333]])

In [498]:
# Forecast
fc = model_fitted.forecast(y=forecast_input, steps=nobs)
df_forecast = pd.DataFrame(fc, index=df_temp1.index[-nobs:], columns=df_temp1.columns + '_forecast')
df_forecast

,HR_forecast,SBP_forecast
timestamp,,
2020-01-06,-5.025033,2.108257
2020-01-07,0.069547,-1.725967
2020-01-08,0.750102,0.057547
2020-01-09,0.496183,-0.775436
2020-01-10,-1.176557,2.282451
2020-01-11,1.101662,-1.755438
2020-01-12,-0.794076,-0.097383


In [345]:
def invert_transformation(df_train, df_forecast, second_diff=False):
    """
    Revert back the differencing to get the forecast to original scale.
    
    df_train   : training set of the timeseries for a particular patient.
    df_forecast: forecast of the timeseries for a particular patient.
    
    output     : forecast dataframe with its values converted to its original scale.
    """
    df_fc = df_forecast.copy()
    columns = df_train.columns
    for col in columns:        
        # Roll back 2nd Diff
        if second_diff:
            df_fc[str(col) + '_forecast'] = (df_train[col].iloc[-1]-df_train[col].iloc[-2]) + df_fc[str(col)+'_forecast'].cumsum()
        # Roll back 1st Diff
        df_fc[str(col) + '_forecast'] = df_train[col].iloc[-1] + df_fc[str(col)+'_forecast'].cumsum()
    return df_fc

In [571]:
df_forecast = invert_transformation(df_train, df_forecast, second_diff=False)

In [572]:
df_forecast

,HR_forecast,SBP_forecast
timestamp,,
2020-01-06,147.894533,271.067517
2020-01-07,220.093547,406.727410
2020-01-08,292.822573,543.055097
2020-01-09,365.244545,678.222270
2020-01-10,436.803971,815.050301
2020-01-11,509.988578,950.675550
2020-01-12,581.576796,1086.844118


In [502]:
def forecast_accuracy(forecast, actual):
    """
     Calculates accuracy by calculating mean absolute percentage error, mean absolute error, 
     and root-mean-squared error.
    
    forecast: forecast dataframe containing the results from a model.
    actual  : test set of a particular patient, with same dates as the forecast dataframe.
    
    output  : mape: mean absolute percentage error
              mae : mean absolute error
              rsme: root mean square error
    """
    forecast = forecast.values
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return mape, mae, rmse

In [698]:
def var_model(patient):
    """
    Creates train and test sets for a particular patient, runs adfuller test to check stationarity, 
    differentiates dataframe if not stationary, fits model, and creates a forecast (to compare with test set), 
    and a future dataframe containing the predicted values for the dates following the test set.
    
    patient: patient uuid
    
    output : df_train2  : copy of the training set for a particular patient.
             df_test    : test set for a particular patient.
             df_forecast: forecasted values to compare with test set.
             df_future  : predicted values for future dates.
    """
    nobs = 7
    diff = 0
    max_diff = 2
    
    df_patient = df_vitals[df_vitals.patient_uuid == patient]
    df_patient.index = pd.DatetimeIndex(df_patient.timestamp).to_period('D')
    df_patient = df_patient[['HR','SBP']].copy()
    df_patient2 = df_patient.copy()
    
    df_train, df_test = df_patient[0:-nobs], df_patient[-nobs:]
    df_train2 = df_train.copy()
    

    are_stationary = []
    for name, column in df_train.iteritems():
        are_stationary.append(adfuller_test(column, name=column.name))
        
    while (all(are_stationary) == False) and (diff < max_diff):
        are_stationary = []
        df_train = df_train.diff().dropna()
        df_patient = df_patient.diff().dropna()
        for name, column in df_train.iteritems():
            are_stationary.append(adfuller_test(column, name=column.name))

        diff += 1

    
    model = VAR(df_train)
    lag_order = 7
    model_fitted = model.fit(lag_order)
    forecast_input = df_train.values[-lag_order:]
    future_input = df_patient.values[-lag_order:]
    
    fc = model_fitted.forecast(y=forecast_input, steps=nobs)
    fc2 = model_fitted.forecast(y=future_input, steps=nobs)
    
    df_forecast = pd.DataFrame(fc, index=df_patient.index[-nobs:], columns=df_patient.columns + '_forecast')
    df_future = pd.DataFrame(fc2, index=df_patient[-nobs:].index + 7, columns = df_patient.columns + '_forecast')
    
    if diff == 1:
        df_forecast = invert_transformation(df_train2, df_forecast, second_diff=False)
        df_future = invert_transformation(df_patient2, df_future, second_diff=False)
    elif diff == 2:
        df_forecast = invert_transformation(df_train2, df_forecast, second_diff=True)
        df_future = invert_transformation(df_patient2, df_future, second_diff=False)
        
    return  df_train2, df_test, df_forecast, df_future
        
        
        

In [701]:
df_train, df_test, df_forecast, df_future = var_model(36279955) # 73939445, 36279955

In [580]:
a = forecast_accuracy(df_forecast, df_test)
a

(HR     0.072291
 SBP    0.056104
 dtype: float64, HR     5.880531
 SBP    7.763567
 dtype: float64, HR     7.399732
 SBP    8.747196
 dtype: float64)

In [508]:
df_vitals[df_vitals.patient_uuid == patients_over_30[7]].tail(10)

,patient_uuid,timestamp,batch,HR,SBP
693,2075791,2019-11-28,0,50.500000,132.500000
694,2075791,2019-11-29,0,53.822345,128.025691
695,2075791,2019-11-30,0,49.500000,128.000000
696,2075791,2019-12-01,0,53.082828,127.983895
697,2075791,2019-12-02,0,52.713070,127.962997
698,2075791,2019-12-03,0,52.343312,127.942099
699,2075791,2019-12-04,0,53.000000,126.000000
700,2075791,2019-12-05,0,56.500000,130.500000
701,2075791,2019-12-06,0,50.666667,122.666667
702,2075791,2019-12-07,0,52.500000,132.500000


In [527]:
# Runs VAR model on all patients and collects data on accuracy.
HR_mape = []
HR_mae = []
HR_rsme = []
SBP_mape = []
SBP_mae = []
SBP_rsme = []

for patient in patients:
     results_train, results_test, results_forecast, results_future = var_model(patient)
    accuracy_indicators = forecast_accuracy(results_forecast, results_test)
    HR_mape.append(accuracy_indicators[0][0])
    HR_mae.append(accuracy_indicators[1][0])
    HR_rsme.append(accuracy_indicators[2][0])
    SBP_mape.append(accuracy_indicators[0][1])
    SBP_mae.append(accuracy_indicators[1][1])
    SBP_rsme.append(accuracy_indicators[2][1])

In [581]:
# Creates and populates dataframe containing data on accuracy of the models for each patient.
df_model = pd.DataFrame(columns = ['patient_uuid', 'HR_mape', 'HR_mae', 'HR_rsme', 'SBP_mape', 'SBP_mae', 'SBP_rsme'])
df_model.patient_uuid = patients
df_model.HR_mape = HR_mape
df_model.HR_mae = HR_mae
df_model.HR_rsme = HR_rsme
df_model.SBP_mape = SBP_mape
df_model.SBP_mae = SBP_mae
df_model.SBP_rsme = SBP_rsme

df_model.head()

,patient_uuid,HR_mape,HR_mae,HR_rsme,SBP_mape,SBP_mae,SBP_rsme
0,1127912,0.056662,5.166311,6.590516,0.042878,5.674134,6.527898
1,1164638,0.015971,1.250469,1.912734,0.016423,1.935242,3.428934
2,1166047,0.114683,7.911892,8.068932,0.126569,14.273312,15.166446
3,1193821,0.029995,1.925800,2.473920,0.103789,11.870040,13.675413
4,1231804,0.065583,5.092933,5.246183,0.126942,14.963018,18.015287


In [551]:
df_model.to_csv('models_stats.csv', index=False)

### Flag-raising algorithm

HR changes 

rising 10 bpm

strict rule bpm > 100; (BPM <55 +/- symptoms)
	
    
BP rising /falling 

		Rising >5mm/30 days
        
		Strict rule 180 mm SBP / under 90mm SBP
        
		Goal is <140mm Hg
        

In [531]:
#df_train, df_test, df_forecast,df_future = var_model(patients_over_30[81])
current = df_train[-23:]
future = df_test
forecast = df_forecast
forecast.columns = ['HR','SBP']
actual = pd.concat([current, future])
pred = pd.concat([current, forecast])

Prediction of need of blood pressure medication or beta blocker medication titration (rule-based):
HR changes 

HR rising/falling  >10bpm/30 days

Strict rule bpm over 100 bpm HR / under 55 bpm HR 
	

BP changes

BP rising /falling  >5mm/30 days

Strict rule 180 mm SBP / under 90mm SBP

Goal is <140mm Hg

In [663]:
def flags(df, report=False):
    """
    Flag-raising algorithm based on health deterioration rules.
    
    df    : 30-day timeseries pandas dataframe.
    report: True if written report requested. False if status of each flag is wanted.
    
    output: returns a list containing the average value of HR and SBP during the last ten days of the time series,
            and the different flags that are raised based on the conditions above.
    """
    rising_HR = False
    lowering_HR = False
    rising_SBP = False
    high_BP = False
    
    first_ten = np.mean(df[:10])
    last_ten = np.mean(df[-10:])

    deltas = (last_ten - first_ten)
    
    if deltas[0] >= 10:
        rising_HR = True
        flag_message_1 = 'Detected rising heart rate. \n'
    elif deltas[0] <= -10:
        lowering_HR = True
        flag_message_1 = 'Detected decreasing heart rate. \n'
    else:
        flag_message_1 = 'No changes in heart rate detected. \n'
        
    if deltas[1] >= 5:
        rising_SBP = True
        flag_message_2 = 'Detected rising blood pressure (SBP). \n'
    else:
        flag_message_2 = 'No changes in blood pressure detected. \n'

    min_HR = np.sum(df.HR < 55)
    max_HR = np.sum(df.HR > 100)

    min_SBP = np.sum(df.SBP < 90)
    max_SBP = np.sum(df.SBP > 180)
    
    flag_message_3 = 'Heart rate lower than 55 bpm detected ' + str(min_HR) + ' times. \n'
    flag_message_4 = 'Heart rate greater than 100 bpm detected ' + str(max_HR) + ' times. \n'
    flag_message_5 = 'Systolic blood pressure (SBP) lower than 90 bpm detected ' + str(min_SBP) + ' times. \n'
    flag_message_6 = 'Systolic blood pressure (SBP) greater than 180 bpm detected ' + str(max_SBP) + ' times. \n'
    flag_message_7 = ''
    
    high_SBP = (df.SBP > 140) & (df.SBP < 179)
    
    if np.sum(high_SBP[-15:]) > 7:
        high_BP = True
        flag_message_7 = 'High blood pressure detected.'
    if report == True:
        if high_BP == True:
            message = flag_message_1 + flag_message_2 + flag_message_3 + flag_message_4 + flag_message_5 + flag_message_6 + flag_message_7 
        else:
            message = flag_message_1 + flag_message_2 + flag_message_3 + flag_message_4 + flag_message_5 + flag_message_6
        print(message)
    else:
        return [last_ten, rising_HR, lowering_HR, rising_SBP, min_HR, max_HR, min_SBP, max_SBP, high_BP]

In [588]:
flags(actual, report=True)

No changes in heart rate detected. 
No changes in blood pressure detected. 
Heart rate lower than 55 bpm detected 0 times. 
Heart rate greater than 100 bpm detected 0 times. 
Systolic blood pressure (SBP) lower than 90 bpm detected 0 times. 
Systolic blood pressure (SBP) greater than 180 bpm detected 0 times. 



In [589]:
flags(pred)

[HR      72.857331
 SBP    138.248167
 dtype: float64, HR      72.419021
 SBP    136.299374
 dtype: float64, False, False, False, 0, 0, 0, 0, False]

In [695]:
def compare_flags(df_train, df_test, df_forecast):
    """
    Compares the results of the flag-raising algorithm.
    
    df_train   : training set of the timeseries for a particular patient.
    df_test    : test set of the timeseries for a particular patient.
    df_forecast: forecast of the timeseries for a particular patient.
    
    output     : list containing HR_error, SBP_error, algorithm accuracy, 
                 predicted flags and actual flags.
    """
    flag_comparison = []
    current = df_train[['HR','SBP']][-23:].copy()
    future = df_test[['HR','SBP']].copy()
    forecast = df_forecast.copy()
    forecast.columns = ['HR', 'SBP']
    actual = pd.concat([current, future])
    pred = pd.concat([current, forecast])
    
    actual_flags = flags(actual)
    pred_flags = flags(pred)
    
    HR_error = np.abs(pred_flags[0][0] - actual_flags[0][0]) / actual_flags[0][0]
    SBP_error = np.abs(pred_flags[0][1] - actual_flags[0][1]) / actual_flags[0][1]

    for i in range(len(actual_flags)):
        flag_comparison.append(actual_flags[i] == pred_flags[i])
    
    algo_accuracy = np.sum(flag_comparison[1:]) / 8
    
    result = [HR_error, SBP_error, algo_accuracy] + pred_flags[1:] + actual_flags[1:]
    return result


In [703]:
compare_flags(df_train, df_test, df_forecast)

[0.05145019158639096,
 0.003688580526220509,
 1.0,
 False,
 False,
 False,
 0,
 0,
 0,
 0,
 False,
 False,
 False,
 False,
 0,
 0,
 0,
 0,
 False]

In [709]:
# Defines column names for dataframe
flag_columns = ['patient_uuid', 'HR_error', 'SBP_error', 'algo_accuracy', 'actual_rising_HR', 'actual_lowering_HR',
                'actual_rising_SBP', 'actual_min_HR', 'actual_max_HR', 'actual_min_SBP', 'actual_max_SBP', 'actual_high_BP',
                'pred_rising_HR', 'pred_lowering_HR', 'pred_rising_SBP', 'pred_min_HR', 'pred_max_HR', 'pred_min_SBP',
                'pred_max_SBP', 'pred_high_BP']

In [710]:
# Creates and populates dataframe based on how the flag-raising algorithm performed against the test and forecast sets.
data = []
for patient in patients:
    df_train, df_test, df_forecast, df_future = var_model(patient)
    flag_row = compare_flags(df_train, df_test, df_forecast)
    flag_row = [patient] + flag_row
    data.append(flag_row)
df_flags = pd.DataFrame(data, columns=flag_columns)

In [729]:
df_flags.to_csv('flag_stats.csv', index=False)

In [713]:
df_flags.head()

,patient_uuid,HR_error,SBP_error,algo_accuracy,actual_rising_HR,actual_lowering_HR,actual_rising_SBP,actual_min_HR,actual_max_HR,actual_min_SBP,actual_max_SBP,actual_high_BP,pred_rising_HR,pred_lowering_HR,pred_rising_SBP,pred_min_HR,pred_max_HR,pred_min_SBP,pred_max_SBP,pred_high_BP
0,1127912,0.021357,0.002203,0.875,False,False,False,0,0,0,0,False,False,False,False,0,1,0,0,False
1,1164638,0.010891,0.008518,1.000,False,False,False,0,0,0,0,False,False,False,False,0,0,0,0,False
2,1166047,0.079817,0.089160,0.875,False,False,False,0,0,1,0,False,False,False,False,0,0,0,0,False
3,1193821,0.013874,0.067252,0.875,False,False,True,0,0,0,0,False,False,False,False,0,0,0,0,False
4,1231804,0.021310,0.088475,1.000,False,False,True,0,0,9,0,False,False,False,True,0,0,9,0,False


In [730]:
np.mean(df_flags.algo_accuracy)

0.9479048697621744